In [2]:
import requests
import pandas as pd

# ── 1. HELPER FUNCTION ────────────────────────────────────────────────────────

def get_ine_table(table_id):
    """Fetch a table from the INE API and return it as JSON."""
    url = f"https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/{table_id}"
    response = requests.get(url)
    response.raise_for_status()
    return response.json()

# ── 2. FETCH TEXTILE WASTE DATA (TABLE 32998) ─────────────────────────────────
# Table 32998: Municipal waste by type and autonomous community
# Format: "07.6 Residuos textiles, Andalucía"
# Region name comes AFTER the comma

raw_waste = get_ine_table(32998)

df_waste = pd.DataFrame([
    {
        "region":             serie["Nombre"],
        "year":               point["NombrePeriodo"],
        "textile_waste_tons": point["Valor"]
    }
    for serie in raw_waste
    for point in serie["Data"]
])

# Filter: keep only textile waste rows
df_waste = df_waste[df_waste["region"].str.contains("07.6", na=False)]

# Extract region name — everything AFTER the comma and space
# "07.6 Residuos textiles, Andalucía" → "Andalucía"
PREFIX = "07.6 Residuos textiles, "
df_waste["region"] = df_waste["region"].str.removeprefix(PREFIX).str.strip()

# Convert year to int for consistent comparison
df_waste["year"] = df_waste["year"].astype(int)

# Find the most recent year and keep only that
max_year_waste = df_waste["year"].max()
print(f"Most recent year in waste data: {max_year_waste}")

df_waste = df_waste[df_waste["year"] == max_year_waste].copy()

# ── 3. FETCH POPULATION DATA (TABLE 67988) ────────────────────────────────────
# Using the SAME year as waste data for consistency

raw_pop = get_ine_table(67988)

df_pop = pd.DataFrame([
    {"region": serie["Nombre"], "year": point["Anyo"], "population": point["Valor"]}
    for serie in raw_pop
    for point in serie["Data"]
])

# Filter: total population only (no breakdown by sex or age)
FILTER_LABEL = "Dato base. Total. Todas las edades. Total."
df_pop = df_pop[df_pop["region"].str.contains(FILTER_LABEL, na=False)]

# Use the SAME year as waste data
df_pop = df_pop[df_pop["year"] == max_year_waste].copy()

# Extract region name — everything before the first dot
# "Andalucía. Dato base. Total..." → "Andalucía"
df_pop["region"] = df_pop["region"].str.extract(r"^([^.]+)\.")

# ── 4. ADD AREA IN KM² ────────────────────────────────────────────────────────
# Source: IGN — https://www.ign.es/web/ign/portal/comunidades-autonomas
# Regional borders have not changed since 1978

AREA_KM2 = {
    "Andalucía":                     87599,
    "Aragón":                        47740,
    "Asturias, Principado de":       10606,
    "Balears, Illes":                 4992,
    "Canarias":                       7446,
    "Cantabria":                      5330,
    "Castilla y León":               94217,
    "Castilla - La Mancha":          79447,
    "Cataluña":                      32113,
    "Comunitat Valenciana":          23265,
    "Extremadura":                   41634,
    "Galicia":                       29584,
    "Madrid, Comunidad de":           8027,
    "Murcia, Región de":             11316,
    "Navarra, Comunidad Foral de":   10392,
    "País Vasco":                     7234,
    "Rioja, La":                      5045,
    "Ceuta":                            20,
    "Melilla":                          14
}

df_pop["area_km2"] = df_pop["region"].map(AREA_KM2)


# ── 5. MERGE ALL THREE DATASETS ───────────────────────────────────────────────

df_merged = df_pop[["region", "year", "population", "area_km2"]].merge(
    df_waste[["region", "textile_waste_tons"]],
    on="region",
    how="left"
)

# ── 6. CALCULATE KEY METRICS ──────────────────────────────────────────────────

df_merged["pop_per_km2"]         = (df_merged["population"] / df_merged["area_km2"]).round(1)
df_merged["waste_kg_per_capita"] = (df_merged["textile_waste_tons"] * 1000 / df_merged["population"]).round(2)

# ── 7. CHECK FOR MISSING VALUES ───────────────────────────────────────────────

missing_area  = df_merged[df_merged["area_km2"].isna()]["region"].tolist()
missing_waste = df_merged[df_merged["textile_waste_tons"].isna()]["region"].tolist()

if missing_area:
    print(f"\nWARNING: No area found for: {missing_area}")
if missing_waste:
    print(f"\nWARNING: No waste data found for: {missing_waste}")
if not missing_area and not missing_waste:
    print("\nAll regions matched successfully")

# ── 8. DISPLAY RESULT ─────────────────────────────────────────────────────────

display(df_merged[[
    "region", "year", "population", "area_km2",
    "pop_per_km2", "textile_waste_tons", "waste_kg_per_capita"
]].reset_index(drop=True))

Most recent year in waste data: 2023



,region,year,population,area_km2,pop_per_km2,textile_waste_tons,waste_kg_per_capita
0,Andalucía,2023,8584147.0,87599,98.0,8616.0,1.00
1,Aragón,2023,1341289.0,47740,28.1,841.0,0.63
2,"Asturias, Principado de",2023,1006060.0,10606,94.9,4125.0,4.10
3,"Balears, Illes",2023,1209906.0,4992,242.4,823.0,0.68
4,Canarias,2023,2213016.0,7446,297.2,493.0,0.22
5,Cantabria,2023,588387.0,5330,110.4,1302.0,2.21
6,Castilla y León,2023,2383703.0,94217,25.3,4459.0,1.87
7,Castilla - La Mancha,2023,2084086.0,79447,26.2,402.0,0.19
8,Cataluña,2023,7901963.0,32113,246.1,14285.0,1.81
9,Comunitat Valenciana,2023,5216195.0,23265,224.2,7327.0,1.40


In [9]:
import plotly.express as px

# ── PREPARE DATA ──────────────────────────────────────────────────────────────
df_plot = df_merged[["region", "waste_kg_per_capita"]].dropna()
df_plot = df_plot.sort_values("waste_kg_per_capita", ascending=True)

# ── ASSIGN COLOR CATEGORY ─────────────────────────────────────────────────────
def assign_category(value):
    if value <= 1:
        return "Low (≤ 1 kg)"
    elif value <= 3:
        return "Medium (1–3 kg)"
    else:
        return "High (> 3 kg)"

df_plot["collection_level"] = df_plot["waste_kg_per_capita"].apply(assign_category)

# ── PLOT ───────────────────────────────────────────────────────────────────────
fig = px.bar(
    df_plot,
    x="waste_kg_per_capita",
    y="region",
    color="collection_level",
    orientation="h",
    color_discrete_map={
        "Low (≤ 1 kg)":    "#e74c3c",
        "Medium (1–3 kg)": "#f39c12",
        "High (> 3 kg)":   "#27ae60"
    },
    labels={
        "waste_kg_per_capita": "Textile waste collected (kg per capita)",
        "region":              "Autonomous Community",
        "collection_level":    "Collection Level"
    },
    title=f"Textile Waste Collection by Region — Spain ({max_year_waste})<br>"
          "<sup>Source: INE Table 32998 — Municipal Waste by Type</sup>",
    text="waste_kg_per_capita"
)

# ── FORMATTING ────────────────────────────────────────────────────────────────
fig.update_traces(
    texttemplate="%{text:.2f} kg",
    textposition="outside"
)

fig.update_layout(
    height=600,
    plot_bgcolor="white",
    legend_title_text="Collection Level",
    xaxis=dict(showgrid=True, gridcolor="#eeeeee"),
    yaxis=dict(showgrid=False),
    margin=dict(l=20, r=100, t=80, b=40)
)

# Reference lines at category boundaries
fig.add_vline(x=1, line_dash="dash", line_color="#e74c3c", opacity=0.5)
fig.add_vline(x=3, line_dash="dash", line_color="#27ae60", opacity=0.5)

fig.show()

In [11]:
import plotly.express as px
import numpy as np

# ── PREPARE DATA ──────────────────────────────────────────────────────────────
df_scatter = df_merged[[
    "region", "area_km2", "waste_kg_per_capita",
    "population", "textile_waste_tons"
]].dropna()

def assign_category(value):
    if value <= 1:
        return "Low (≤ 1 kg)"
    elif value <= 3:
        return "Medium (1–3 kg)"
    else:
        return "High (> 3 kg)"

df_scatter["collection_level"] = df_scatter["waste_kg_per_capita"].apply(assign_category)

# ── АВТОМАТИЧЕСКИЙ ВЫБОР ПОДПИСЕЙ ─────────────────────────────────────────────
top_y = set(df_scatter.nlargest(3, "waste_kg_per_capita")["region"])
top_x = set(df_scatter.nlargest(3, "area_km2")["region"])
label_regions = top_y | top_x  # объединение — уникальные регионы

df_scatter["label"] = df_scatter["region"].where(
    df_scatter["region"].isin(label_regions), ""
)

# ── STATS ─────────────────────────────────────────────────────────────────────
national_avg = (
    df_scatter["textile_waste_tons"].sum() * 1000
    / df_scatter["population"].sum()
)
corr = df_scatter["area_km2"].corr(df_scatter["waste_kg_per_capita"])

# ── TREND LINE (OLS) ──────────────────────────────────────────────────────────
m, b = np.polyfit(df_scatter["area_km2"], df_scatter["waste_kg_per_capita"], 1)
x_range = np.linspace(df_scatter["area_km2"].min(), df_scatter["area_km2"].max(), 200)

# ── PLOT ──────────────────────────────────────────────────────────────────────
fig = px.scatter(
    df_scatter,
    x="area_km2",
    y="waste_kg_per_capita",
    color="collection_level",
    text="label",
    color_discrete_map={
        "Low (≤ 1 kg)":    "#e74c3c",
        "Medium (1–3 kg)": "#f39c12",
        "High (> 3 kg)":   "#27ae60"
    },
    custom_data=["population", "textile_waste_tons", "region"],
    labels={
        "area_km2":            "Region area (km²)",
        "waste_kg_per_capita": "Textile waste collected (kg per capita)",
        "collection_level":    "Collection Level"
    },
    title=(
        f"Larger Regions Collect Less Textile Waste per Capita — Spain ({max_year_waste})<br>"
        f"<sup>OLS trend r = {corr:.2f} · Source: INE Table 32998, IGN</sup>"
    )
)

# ── TOOLTIP ───────────────────────────────────────────────────────────────────
fig.update_traces(
    textposition="top center",
    textfont=dict(size=11, color="#444444"),
    marker=dict(size=12, line=dict(color="white", width=1)),
    hovertemplate=(
        "<b>%{customdata[2]}</b><br>"
        "Area: %{x:,.0f} km²<br>"
        "Waste: %{y:.2f} kg/capita<br>"
        "Population: %{customdata[0]:,.0f}<br>"
        "Total collected: %{customdata[1]:,.0f} t"
        "<extra></extra>"
    )
)

# ── TREND LINE ────────────────────────────────────────────────────────────────
fig.add_scatter(
    x=x_range,
    y=m * x_range + b,
    mode="lines",
    line=dict(color="#7f8c8d", dash="dot", width=2),
    name=f"Trend (r = {corr:.2f})",
    hoverinfo="skip"
)

# ── NATIONAL AVERAGE ──────────────────────────────────────────────────────────
fig.add_hline(
    y=national_avg,
    line_dash="dot",
    line_color="#2c3e50",
    opacity=0.7,
    annotation_text=f"Spain avg: {national_avg:.2f} kg",
    annotation_position="right",
    annotation_font_size=12
)

# ── CATEGORY BOUNDARIES ───────────────────────────────────────────────────────
fig.add_hline(y=1, line_dash="dash", line_color="#e74c3c", opacity=0.25)
fig.add_hline(y=3, line_dash="dash", line_color="#27ae60", opacity=0.25)

# ── LAYOUT ────────────────────────────────────────────────────────────────────
fig.update_layout(
    height=650,
    plot_bgcolor="white",
    legend_title_text="Collection Level",
    xaxis=dict(showgrid=True, gridcolor="#eeeeee", zeroline=False),
    yaxis=dict(showgrid=True, gridcolor="#eeeeee", zeroline=False),
    margin=dict(l=20, r=140, t=90, b=40),
    font=dict(family="Arial", size=12)
)

fig.show()
print(f"Weighted national average: {national_avg:.3f} kg/capita")
print(f"Pearson r (area vs waste/capita): {corr:.3f}")
print(f"Labelled regions: {sorted(label_regions)}")

Weighted national average: 1.483 kg/capita
Pearson r (area vs waste/capita): -0.313
Labelled regions: ['Andalucía', 'Asturias, Principado de', 'Castilla - La Mancha', 'Castilla y León', 'Navarra, Comunidad Foral de', 'Rioja, La']


In [ ]:
import requests
import pandas as pd

# Запрос к API Humana
url = "https://hcontainerdb.humana-spain.org/api/sites/?format=json&country=1"
response = requests.get(url)
data = response.json()

print(f"Всего контейнеров: {len(data)}")

# Строим DataFrame
df = pd.DataFrame(data)

# Разделяем geolocation на lat и lon
df[["lat", "lon"]] = df["geolocation"].str.split(",", expand=True).astype(float)

# Смотрим структуру
print(df.head())
print("\nВсе поля:", df.columns.tolist())
print("\nПо городам (топ 10):")
print(df["city"].value_counts().head(10))

Всего контейнеров: 5103
      id                                            name  \
0   2928       10 Partida Barranquets Cl-29a Els Poblets   
1  11557                 11 C. Formentera de Bellreguard   
2   1974              172 Carrer de les Eres Rafelguaraf   
3   2913  18 C. Jorge de Austria de Villar del Arzobispo   
4   2181                        1 C/ Montduber de Xeraco   

                              address                  city  \
0  Partida Barranquets Cl-29a, 45-205           Els Poblets   
1                    11 C. Formentera           Bellreguard   
2              172 Carrer de les Eres           Rafelguaraf   
3            Avenida Jorge de Austria  Villar del Arzobispo   
4                      1 C/ Montduber                Xeraco   

                     geolocation  category        lat       lon  
0           38.8553984,0.0105988         4  38.855398  0.010599  
1          38.9638037,-0.1316462         4  38.963804 -0.131646  
2  39.045155,-0.4684371000000001      